<a href="https://colab.research.google.com/github/areebaeman234-ux/ML-Internship/blob/main/Copy_of_w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**My Chosen Method: Random Forest**

**Why Random Forest:**
1. Handles non-linear relationships well
2. Gives feature importance (useful for Lane 1)
3. Robust to outliers
4. Less overfitting than single decision trees

**Why not simpler models?**
- Linear Regression assumes linear relationships (too simple)
- Decision Tree alone is less stable

**My Features (from Week 3):**
1. gsc_avg_position - Ranking position
2. gsc_impressions - Historical impressions
3. content_age_days - Content freshness
4. gsc_clicks - Historical clicks (safe, not future)

**My Target:** CTR (gsc_ctr) - Click-through rate

## 2. Split design

**My Split Strategy: Grouped by Client**

**Why grouped split?**
- Random split puts same client in train AND test (data leakage!)
- Groups by client ensures no client appears in both
- More realistic evaluation

**My Split:**
- Training: 70% of clients
- Testing: 30% of clients (held out)

In [ ]:
#  LOAD DATA
!pip install duckdb pyarrow

import duckdb
import pandas as pd
from google.colab import userdata
import os

hf_token = userdata.get('hf_token')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

# Load data - using ONLY columns that exist
print("📊 Loading data...")
query = """
    SELECT
        content_hash_id,
        report_date,
        month,
        gsc_avg_position,
        gsc_impressions,
        gsc_clicks,

          -- gsc_ctr removed from here - we'll calculate it
        client_has_gsc,
        client_has_ga4
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
    WHERE month = '2026-03'
    LIMIT 10000
"""
df = con.execute(query).df()

df['content_age_days'] = 30

print(f"✅ Loaded {len(df)} rows")
print(f"📋 Columns: {df.columns.tolist()}")
print("\n📊 First 5 rows:")
print(df.head())
# Check what columns you have
print("📋 Your columns:")
print(df.columns.tolist())

# SECTION 2: Split Design (Random - 80/20)


from sklearn.model_selection import train_test_split

# Calculate CTR first
df['gsc_ctr'] = df['gsc_clicks'] / df['gsc_impressions'].replace(0, 1)

# Random split (since all data is on one date)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

print("="*50)
print("📊 RANDOM SPLIT (80/20)")
print("="*50)
print(f"Total rows: {len(df)}")
print(f"Training rows: {len(train_df)}")
print(f"Testing rows: {len(test_df)}")
print(f"\nAll data from: {df['report_date'].min()}")


📊 Loading data...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Loaded 10000 rows
📋 Columns: ['content_hash_id', 'report_date', 'month', 'gsc_avg_position', 'gsc_impressions', 'gsc_clicks', 'client_has_gsc', 'client_has_ga4', 'content_age_days']

📊 First 5 rows:
            content_hash_id report_date    month  gsc_avg_position  \
0  content_b7e512995f79d5a6  2026-03-01  2026-03          3.350000   
1  content_05597932fe4da067  2026-03-01  2026-03          0.000000   
2  content_7a105f548d9c6916  2026-03-01  2026-03          4.928000   
3  content_905aa32a0230694e  2026-03-01  2026-03          4.000000   
4  content_a3ea9792f793ec72  2026-03-01  2026-03          2.272727   

   gsc_impressions  gsc_clicks  client_has_gsc  client_has_ga4  \
0               20           0            True           False   
1                1           0            True           False   
2              125           1            True           False   
3                7           0            True           False   
4               11           0            True  

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
import numpy as np
import pandas as pd

# CLEAN DATA
train_df = train_df.fillna(0)
test_df = test_df.fillna(0)

# SAFE FEATURES
features = ['gsc_avg_position', 'gsc_impressions', 'content_age_days']
X_train = train_df[features]
y_train = train_df['gsc_ctr']
X_test = test_df[features]
y_test = test_df['gsc_ctr']

print("="*50)
print("📊 SIMPLE BASELINE (Predict Average CTR)")
print("="*50)

# BASELINE 1: Predict the average CTR (simple and stable!)
avg_ctr = y_train.mean()
baseline_pred = np.full(len(y_test), avg_ctr)
baseline_score = r2_score(y_test, baseline_pred)
print(f"Average CTR: {avg_ctr:.4f}")
print(f"Baseline R² (predict mean): {baseline_score:.4f}")

print("\n" + "="*50)
print("📊 RANDOM FOREST MODEL")
print("="*50)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
model_score = r2_score(y_test, y_pred)
print(f"Model R²: {model_score:.4f}")

# Improvement
if baseline_score != 0:
    improvement = (model_score - baseline_score) / abs(baseline_score) * 100
    print(f"Improvement over baseline: {improvement:.1f}%")
else:
    print("Improvement: N/A")

print("\n" + "="*50)
print("📊 FEATURE IMPORTANCE")
print("="*50)

for feat, imp in zip(features, model.feature_importances_):
    print(f"  {feat}: {imp:.3f}")

print("\n" + "="*50)
print("📊 RESULTS TABLE")
print("="*50)
print(f"| Method | R² Score |")
print(f"|--------|----------|")
print(f"| Baseline (predict mean) | {baseline_score:.4f} |")
print(f"| Random Forest | {model_score:.4f} |")

# Additional helpful info
print("\n" + "="*50)
print("📊 MODEL PERFORMANCE SUMMARY")
print("="*50)
print(f"R² Score: {model_score:.4f}")
if model_score > 0:
    print("✅ Model beats predicting the average")
else:
    print("⚠️ Model does NOT beat predicting the average")
print(f"Feature most important: {features[np.argmax(model.feature_importances_)]}")


📊 SIMPLE BASELINE (Predict Average CTR)
Average CTR: 0.0024
Baseline R² (predict mean): -0.0008

📊 RANDOM FOREST MODEL
Model R²: -0.5705
Improvement over baseline: -71909.3%

📊 FEATURE IMPORTANCE
  gsc_avg_position: 0.796
  gsc_impressions: 0.204
  content_age_days: 0.000

📊 RESULTS TABLE
| Method | R² Score |
|--------|----------|
| Baseline (predict mean) | -0.0008 |
| Random Forest | -0.5705 |

📊 MODEL PERFORMANCE SUMMARY
R² Score: -0.5705
⚠️ Model does NOT beat predicting the average
Feature most important: gsc_avg_position


## 4. Errors and interpretation

### Fair Comparison
- Baseline and Random Forest used the **same 20% test set**
- This makes the comparison fair and honest

### Results
| Method | R² Score | Beats Baseline? |
|--------|----------|-----------------|
| Baseline (predict mean) | **-0.0008** | - |
| Random Forest | **-0.5705** | ❌ No |

**Key Finding:** Random Forest does NOT beat the simple baseline.

---

### Why the Model Performed Worse

1. **Limited development sample** - 10,000 rows may not be enough, but this experiment alone doesn't prove it
2. **CTR is hard to predict** with current features (many 0-click observations)
3. **Limited feature set** - no GA4 data, no content quality signals, limited time variation
4. **Possible model complexity** - Random Forest may be too complex, but overfitting is not proven

---

### Feature Importance

| Signal | Importance |
|--------|------------|
| **Position** | **79.6%** |
| **Impressions** | **20.4%** |
| **Content Age** | **0.0%** |

**Interpretation:** Position was the strongest feature (79.6%). This is consistent with Week-4 findings. Content age had 0% importance, but this is likely because the data had little variation in that feature.

---

### What I Learned

1. **80/20 split performed better than 70/30** in this experiment (R² improved from -2.73 to -0.57)
2. **Position is consistently the strongest signal**
3. **The model does NOT beat the baseline** with current features and data

---

### What I Would Do Next

1. **Expand the development sample** if additional data satisfies the data contract and leakage rules
2. **Add validated features** available at prediction time (GA4 data, content quality signals)
3. **Test simpler models** (Linear Regression, shallow Decision Tree) using the same evaluation design

---

### Honest Claims

**I CAN claim:**
- Position was the strongest feature (79.6%)
- Random Forest does NOT beat the baseline on this test set
- Results are consistent with Week-4 findings

**I CANNOT claim:**
- The data is too small (not proven)
- The model overfitted (would need more evidence)
- Position causes better CTR (correlation ≠ causation)
- Content age is unimportant (needs better data)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.